In [1]:
import torch
import time
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoProcessor,
    CLIPModel,
    CLIPProcessor
)
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice
from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt

# Configuration
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
DATASET_NAME = "arampacha/rsicd"
NUM_SOFT_TOKENS = 16
LEARNING_RATE = 1e-3
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
EPOCHS = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:
from transformers import AutoModel
model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True  # Required for Qwen models
)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Freeze model parameters
for param in model.parameters():
    param.requires_grad = False

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

In [4]:
class CoOpAdapter(torch.nn.Module):
    def __init__(self, model, num_tokens=16):
        super().__init__()
        self.model = model
        self.num_tokens = num_tokens
        self.soft_prompt = torch.nn.Parameter(
            torch.randn(1, num_tokens, model.config.hidden_size, 
                        dtype=torch.bfloat16)
        )
    
    def forward(self, pixel_values, input_ids=None, attention_mask=None):
        # Extract visual features
        vision_outputs = self.model.vision_model(pixel_values=pixel_values)
        visual_features = vision_outputs.last_hidden_state
        
        # Prepare soft prompt
        batch_size = visual_features.size(0)
        soft_prompt = self.soft_prompt.expand(batch_size, -1, -1)
        
        # Combine soft prompt + visual features
        inputs_embeds = torch.cat([soft_prompt, visual_features], dim=1)
        
        # Language model forward pass
        outputs = self.model.language_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            input_ids=input_ids
        )
        return outputs

coop_adapter = CoOpAdapter(model, NUM_SOFT_TOKENS).to(DEVICE)
optimizer = torch.optim.Adam(coop_adapter.parameters(), lr=LEARNING_RATE)

In [7]:
def process_example(example):
    # Simplified processing for RSICD
    image = example["image"].convert("RGB")
    caption = example["captions"][0]  # Use first caption
    
    # Create model input - separate image and text
    text = "<|im_start|>user\n<image>\nDescribe this remote sensing image in detail.<|im_end|>\n<|im_start|>assistant\n"
    
    # Process with Qwen-specific processor
    processed = processor(
        text=text,
        images=image,
        return_tensors="pt",
        padding=True
    )
    
    return {
        "pixel_values": processed["pixel_values"].squeeze(0),
        "input_ids": processed["input_ids"].squeeze(0),
        "attention_mask": processed["attention_mask"].squeeze(0),
        "caption": caption
    }

In [8]:
dataset = load_dataset(DATASET_NAME)
train_ds = dataset["train"].map(process_example, remove_columns=["image", "captions"])
test_ds = dataset["test"].map(process_example, remove_columns=["image", "captions"])

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

Map:   0%|          | 0/1093 [00:00<?, ? examples/s]

OSError: [Errno 28] No space left on device

In [ ]:
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "input_ids": torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"] for x in batch], 
            batch_first=True, 
            padding_value=tokenizer.pad_token_id
        ),
        "attention_mask": torch.nn.utils.rnn.pad_sequence(
            [x["attention_mask"] for x in batch], 
            batch_first=True, 
            padding_value=0
        ),
        "captions": [x["caption"] for x in batch]
    }

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=1, collate_fn=collate_fn)